In [1]:
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.spatial.distance import pdist, squareform, mahalanobis

In [2]:
pca_data_abck = pd.read_csv('../data/results/pca_data_abck.csv')
display(pca_data_abck.head())

data_fixed = pd.read_csv('./results/data_fixed.csv')
display(data_fixed.head())

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,cluster_abck
0,-2.173814,0.560687,-0.244653,-0.000022,-0.198642,-0.297102,-0.449977,-0.315806,-0.283185,-0.048349,3
1,-2.135326,0.504553,-0.095536,-0.305302,-0.183530,0.056558,-0.207734,-0.030956,0.008387,-0.057170,3
2,-2.165984,0.458176,-0.002323,-0.326949,-0.166642,0.099856,-0.208867,0.031741,0.089761,0.017689,3
3,-2.211613,0.503966,-0.129927,-0.042172,-0.189566,-0.223255,-0.443616,-0.230362,-0.179810,0.050264,3
4,-2.324292,0.460397,-0.136303,0.202740,-0.166470,-0.296635,-0.488494,-0.353747,-0.337112,0.184335,3


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc,fecha,referencia
0,0.5,1.00,3000.0,3000.0,2.0,0.68,11.81,1.883,0.05285,4.0,8.20,10.0,41.0,313.0,0.13530,04/06/2001,P1
1,0.5,1.00,2200.0,7000.0,2.0,0.18,23.42,5.179,0.16410,4.0,8.36,10.0,185.0,453.0,0.09056,04/06/2001,P2
2,0.5,1.00,7000.0,17000.0,2.0,0.48,19.68,6.043,0.20470,4.0,8.47,10.0,209.0,470.0,0.09999,04/06/2001,P3
3,0.5,1.00,11000.0,11000.0,2.0,0.58,7.87,3.159,0.09761,4.0,8.34,10.0,74.0,350.0,0.09698,04/06/2001,P4
4,0.5,2.89,13000.0,13000.0,2.0,0.35,7.75,1.093,0.03675,5.2,8.44,10.0,0.0,293.0,0.11720,04/07/2001,P1


In [3]:
data_fixed["cluster"] = pca_data_abck['cluster_abck'].copy()
data_fixed.head()

,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc,fecha,referencia,cluster
0,0.5,1.00,3000.0,3000.0,2.0,0.68,11.81,1.883,0.05285,4.0,8.20,10.0,41.0,313.0,0.13530,04/06/2001,P1,3
1,0.5,1.00,2200.0,7000.0,2.0,0.18,23.42,5.179,0.16410,4.0,8.36,10.0,185.0,453.0,0.09056,04/06/2001,P2,3
2,0.5,1.00,7000.0,17000.0,2.0,0.48,19.68,6.043,0.20470,4.0,8.47,10.0,209.0,470.0,0.09999,04/06/2001,P3,3
3,0.5,1.00,11000.0,11000.0,2.0,0.58,7.87,3.159,0.09761,4.0,8.34,10.0,74.0,350.0,0.09698,04/06/2001,P4,3
4,0.5,2.89,13000.0,13000.0,2.0,0.35,7.75,1.093,0.03675,5.2,8.44,10.0,0.0,293.0,0.11720,04/07/2001,P1,3


In [4]:
# Calcular el perfil químico de cada cluster
perfil_quimico = data_fixed.drop(columns=['fecha', 'referencia']).groupby('cluster').mean()
perfil_quimico['frequencia'] = data_fixed.groupby('cluster').size()
perfil_quimico.to_csv('../data/results/perfil_quimico.csv', sep=',')
display(perfil_quimico)

,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc,frequencia
cluster,,,,,,,,,,,,,,,,
0,13.966431,86.989688,1.628583e+05,3.471914e+05,403.959232,77.722049,17.834615,32.111604,3.968332,326.761713,7.934241,92.101463,346.870573,163.965673,1.878697,274
1,26.250000,48.080000,8.200000e+06,1.400000e+07,67.737000,3.637500,28.741500,90.115000,27.220000,152.214000,7.275000,77.550000,364.840000,24.892900,0.563830,2
2,102.043158,105.680358,1.471876e+05,4.107169e+05,504.516211,42.929211,95.579221,116.047760,33.445580,184.780053,6.425368,311.528484,227.832926,34.167884,7.272504,19
3,10.860408,19.371700,2.113459e+04,3.698176e+04,62.850074,19.078973,35.741692,7.480308,0.433990,78.807849,8.082840,59.979817,170.166612,230.201245,1.218391,345


In [5]:
from pathlib import Path


def empty_directory(directory_path):
    path = Path(directory_path)
    if path.is_dir():
        return not any(path.iterdir())
    else:
        raise ValueError("This path is not a directory")
    

def pfqb_graph_chem(data_fixed: pd.DataFrame, output_path:str) -> None:

    directory_path = Path(output_path).parent

    try:
        if empty_directory(directory_path):
            column_list = data_fixed.drop(columns=['fecha', 'referencia', 'cluster']).columns.to_list()

            col_size = len(column_list)

            n_cols = 3
            n_rows = int(col_size/n_cols)

            i_col = 0
            i_row = 0

            for col_1 in column_list:
                i_row = 0
                fig, ax = plt.subplots(
                    nrows=n_rows, 
                    ncols=n_cols, 
                    figsize=(3*n_rows, 5*n_cols)
                )
                for col_2 in column_list:
                    # if col_1 != col_2:
                    sns.scatterplot(
                        data=data_fixed, 
                        x=col_1, 
                        y=col_2, 
                        hue='cluster', 
                        palette='viridis', 
                        s=50, 
                        alpha=1,
                        edgecolor='w',
                        ax=ax[i_row, i_col]
                    )

                    # Personalización de títulos (basado en tu interpretación de loadings)
                    ax[i_row, i_col].set_title(f'{col_1} vs {col_2}', fontsize=15)
                    ax[i_row, i_col].set_xlabel(f'{col_1}', fontsize=12)
                    ax[i_row, i_col].set_ylabel(f'{col_2}', fontsize=12)
                    ax[i_row, i_col].legend(title='Clusters', loc='best')
                    ax[i_row, i_col].grid(True, linestyle='--', alpha=0.50)

                    # actualizacion contadores ax
                    if i_row < n_rows-1:
                        i_row += 1
                    else:
                        i_row = 0

                    if i_col < n_cols-1:
                        i_col += 1
                    else:
                        i_col = 0

                
                plt.tight_layout()
                plt.savefig(f'{directory_path}comp_{col_1}_clusters.png')
                plt.show()
        else:
            print(f"This directory {directory_path} is not empty. Choose another one.")
    except ValueError as e:
        print(e)

In [6]:
pfqb_graph_chem(data_fixed, './graphs/results/')

This directory graphs is not empty. Choose another one.


In [7]:
euclidean_distance = pdist(perfil_quimico.values, metric='euclidean')
euclidean_dist_matrix = squareform(euclidean_distance)

print(euclidean_dist_matrix)

[[       0.         15842816.38269348    65431.41558966   341051.06872224]
 [15842816.38269348        0.         15796088.2081582  16182080.15312748]
 [   65431.41558966 15796088.2081582         0.           394420.8239329 ]
 [  341051.06872224 16182080.15312748   394420.8239329         0.        ]]


In [8]:
manhattan_distance = pdist(perfil_quimico.values, metric='cityblock')
manhattan_dist_matrix = squareform(manhattan_distance)
print(manhattan_dist_matrix)

[[       0.         21691124.12447875    80501.75323149   453044.67964797]
 [21691124.12447875        0.         21643241.35066158 22142899.92140128]
 [   80501.75323149 21643241.35066158        0.           501577.50496938]
 [  453044.67964797 22142899.92140128   501577.50496938        0.        ]]


In [9]:
cov_matrix = np.cov(data_fixed.drop(['fecha', 'referencia'], axis=1).values.T)
inv_cov_matrix = np.linalg.pinv(cov_matrix)

n_centroides = perfil_quimico.shape[0]
mahalanobis_matrix = np.zeros([n_centroides, n_centroides])

for i in range(n_centroides):
    for j in range(n_centroides):
        if i != j:
            mahalanobis_matrix[i, j] = mahalanobis(
                perfil_quimico.iloc[i].values,
                perfil_quimico.iloc[j].values,
                inv_cov_matrix
            )

mahalanobis_dist_matrix = (mahalanobis_matrix + mahalanobis_matrix.T) / 2
np.fill_diagonal(mahalanobis_dist_matrix, 0)

print(mahalanobis_dist_matrix)

[[  0.         334.39257833 314.75124203  84.8399797 ]
 [334.39257833   0.          25.06721938 419.14876246]
 [314.75124203  25.06721938   0.         399.56823058]
 [ 84.8399797  419.14876246 399.56823058   0.        ]]


In [10]:
print("ESTADÍSTICAS DE SEPARACIÓN:")
print(f"  - Distancia mínima (centroides más cercanos): {np.min(euclidean_distance[euclidean_distance > 0]):.4f}")
print(f"  - Distancia máxima (centroides más lejanos): {np.max(euclidean_distance):.4f}")
print(f"  - Distancia promedio: {np.mean(euclidean_distance):.4f}")
print(f"  - Coeficiente de variación: {np.std(euclidean_distance)/np.mean(euclidean_distance):.4f}")

ESTADÍSTICAS DE SEPARACIÓN:
  - Distancia mínima (centroides más cercanos): 65431.4156
  - Distancia máxima (centroides más lejanos): 16182080.1531
  - Distancia promedio: 8103648.0087
  - Coeficiente de variación: 0.9673


# Análisis de separación
Con un coficiente de variacion de 0.9673, se comprende los siguiente:
- Alta dispersión relativa: Las distancias entre los pares de centroides son muy heterogéneas. Un CV cercano a 1 (o 100%) indica que la desviación estándar de las distancias es casi tan grande como la media de esas distancias.
- Separación desigual: Algunos pares de centroides están muy cercanos entre sí, mientras que otros están muy alejados
- Estructura asimétrica: Los 4 clústeres no están distribuidos uniformemente en el espacio de 16 parámetros químicos
- Posible jerarquía: Puede haber 2 centroides muy próximos (formando un super-grupo) y otros 2 más aislados